<a href="https://colab.research.google.com/github/SarikMaj/shapely-project/blob/main/shapely_projekt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Knihovna Shapely

**Semestrální projekt – představení knihovny**

Pracujeme s otevřenými daty **Natural Earth**, která pokrývají celý svět (státy, regiony, sídla, řeky). Většinu ukázek zaměříme na **Českou republiku**, protože výsledky tu snadno ověříme: spočítáme rozlohy krajů, zjistíme, které kraje spolu sousedí, ve kterém kraji leží které město, jak daleko jsou města od hranice nebo která leží u řeky. Na závěr zpracujeme sídla celého světa najednou.

## Co je Shapely?

**Shapely** je knihovna pro práci s **geometrií** – body, liniemi a polygony. Umožňuje:

* vytvářet geometrie a zjišťovat jejich vlastnosti (plocha, délka,..),
* testovat **prostorové vztahy** (leží uvnitř? dotýká se? protíná?),
* provádět **geometrické operace** (průnik, sjednocení, rozdíl, buffer,..).

Uvnitř stojí na knihovně **GEOS**, kterou používají i **QGIS** a **PostGIS** – výsledky jsou proto stejné jako v těchto nástrojích. Shapely je také základem knihovny **GeoPandas**: každý prvek ve sloupci `geometry` je objekt Shapely.

In [ ]:
import pandas as pd                       # práce s tabulkami
import matplotlib.pyplot as plt           # kreslení map a grafů
import geopandas as gpd                   # načítání prostorových dat
import shapely                            # knihovna, kterou představujeme
from shapely import LineString            # typ geometrie "linie" (použijeme pro spojnice)
from shapely.ops import nearest_points    # hledání nejbližších bodů dvou geometrií
from shapely.plotting import plot_polygon, plot_line, plot_points   # vykreslení geometrií

print("Verze Shapely:", shapely.__version__)

## 1. Data

Použijeme otevřená data **[Natural Earth](https://www.naturalearthdata.com/)** v měřítku 1 : 10 mil.:

| Vrstva | Obsah | Typ geometrie |
|---|---|---|
| `ne_10m_admin_0_countries` | státy světa | polygony |
| `ne_10m_admin_1_states_provinces` | kraje (a regiony dalších států) | polygony |
| `ne_10m_populated_places_simple` | významná sídla světa | body |
| `ne_10m_rivers_lake_centerlines`, `ne_10m_rivers_europe` | řeky | linie |

Data načteme přes GeoPandas přímo z GitHubu projektu Natural Earth. Každá vrstva je tabulka, ve které sloupec `geometry` obsahuje objekty Shapely.

In [ ]:
# Načtení dat přímo z GitHubu Natural Earth (formát GeoJSON)
staty = gpd.read_file("https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_10m_admin_0_countries.geojson")
regiony = gpd.read_file("https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_10m_admin_1_states_provinces.geojson")
mesta_svet = gpd.read_file("https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_10m_populated_places_simple.geojson")

# řeky jsou ve dvou souborech (hlavní řeky světa + evropské řeky) – spojíme je do jedné tabulky
reky_svet = pd.concat([
    gpd.read_file("https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_10m_rivers_lake_centerlines.geojson"),
    gpd.read_file("https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_10m_rivers_europe.geojson"),
])

print(f"Států: {len(staty)}, regionů: {len(regiony)}, sídel: {len(mesta_svet)}, úseků řek: {len(reky_svet)}")

### Výběr České republiky a převod do metrů

Data jsou v zeměpisných souřadnicích (**stupních**). Shapely počítá v rovině a v jednotkách souřadnic – plocha by tedy vyšla ve „čtverečních stupních“. Proto data ČR převedeme do českého souřadnicového systému **S-JTSK (EPSG:5514)**, který je v **metrech**. Převod udělá GeoPandas funkcí `to_crs`.

In [ ]:
# hranice ČR – vybereme řádek s kódem státu CZE, převedeme do metrů a vezmeme samotný polygon
cr = staty[staty["ADM0_A3"] == "CZE"].to_crs("EPSG:5514").geometry.iloc[0]

# 14 krajů ČR
kraje = regiony[regiony["adm0_a3"] == "CZE"].to_crs("EPSG:5514").reset_index(drop=True)

# 12 největších českých měst
mesta = mesta_svet[mesta_svet["adm0_a3"] == "CZE"].to_crs("EPSG:5514").reset_index(drop=True)

# řeky – nejdřív vybereme jen ty v okolí ČR (obdélník 12–19° v. d., 48–52° s. š.), pak je převedeme
reky = reky_svet.cx[12:19, 48:52].to_crs("EPSG:5514")

# v datech jsou některé názvy anglicky nebo s chybou – opravíme je
kraje["name"] = kraje["name"].replace({"Prague": "Praha"})
mesta["name"] = mesta["name"].replace({"Prague": "Praha", "Pizen": "Plzeň", "Usti Nad Labem": "Ústí nad Labem"})

print(f"Krajů: {len(kraje)}, měst: {len(mesta)}, úseků řek v okolí ČR: {len(reky)}")

## 2. Typy geometrie

Shapely rozlišuje několik typů geometrie. Základní jsou **bod** (`Point`), **linie** (`LineString`) a **polygon** (`Polygon`). Ukážeme si je na našich datech: Česká republika je polygon, Vltava linie a Praha bod.

In [ ]:
# Vltava je v datech rozdělená na několik úseků – funkcí union_all je sloučíme do jednoho objektu
vltava = shapely.union_all(reky.loc[reky["name"] == "Vltava", "geometry"])

# Praha jako bod z tabulky měst
praha = mesta.loc[mesta["name"] == "Praha", "geometry"].iloc[0]

# geom_type vrací typ geometrie
print("Česká republika:", cr.geom_type)
print("Vltava:         ", vltava.geom_type, " (více linií v jednom objektu)")
print("Praha:          ", praha.geom_type)

# vypsaná geometrie se zobrazí ve formátu WKT (Well-Known Text), souřadnice jsou v metrech
print("\nPraha ve formátu WKT:", praha)

In [ ]:
# vykreslení všech tří typů do jedné mapy
fig, ax = plt.subplots(figsize=(10, 6))
plot_polygon(cr, ax=ax, add_points=False, color="lightgrey", edgecolor="black")   # polygon
plot_line(vltava, ax=ax, add_points=False, color="tab:blue", linewidth=2)         # linie
plot_points(praha, ax=ax, color="tab:red", markersize=10)                          # bod
ax.set_title("Polygon (ČR), linie (Vltava) a bod (Praha)")
ax.set_aspect("equal")    # stejné měřítko na obou osách, aby mapa nebyla deformovaná
ax.axis("off")            # skryje osy s čísly
plt.show()

## 3. Rozlohy krajů

Funkce `shapely.area` spočítá plochu a `shapely.length` délku hranice – pro všechny kraje najednou.

In [ ]:
# plochu převedeme z m² na km² (dělení milionem), délku z m na km (dělení tisícem)
kraje["rozloha_km2"] = (shapely.area(kraje.geometry) / 1_000_000).round()
kraje["obvod_km"] = (shapely.length(kraje.geometry) / 1_000).round()

print(f"Rozloha celé ČR: {cr.area / 1_000_000:,.0f} km² (oficiálně 78 871 km²)")

# tabulka seřazená od největšího kraje
kraje[["name", "rozloha_km2", "obvod_km"]].sort_values("rozloha_km2", ascending=False)

## 4. Které kraje spolu sousedí?

Prostorové vztahy testují **predikáty** – funkce, které vracejí `True` (ano) nebo `False` (ne):

| Funkce | Otázka |
|---|---|
| `intersects` | mají něco společného? |
| `touches` | dotýkají se jen hranicí? |
| `contains` | leží jedno uvnitř druhého? |

Sousední kraje se dotýkají hranicí, použijeme tedy `touches`.

In [ ]:
# pro každý kraj zjistíme, kterých krajů se dotýká
for nazev, tvar in zip(kraje["name"], kraje.geometry):
    dotyka_se = shapely.touches(tvar, kraje.geometry)      # True/False pro všech 14 krajů
    sousedi = kraje["name"][dotyka_se]                       # názvy krajů, kde vyšlo True
    print(f"{nazev}: {', '.join(sousedi)}")

## 5. Ve kterém kraji leží které město?

Použijeme predikát `contains` – zjistí, který kraj město (bod) obsahuje.

In [ ]:
for nazev, bod in zip(mesta["name"], mesta.geometry):
    obsahuje = shapely.contains(kraje.geometry, bod)       # True u kraje, ve kterém město leží
    kraj = kraje["name"][obsahuje].iloc[0]                  # název toho kraje
    print(f"{nazev:18} → {kraj}")

## 6. Jak daleko jsou města od státní hranice?

* `cr.boundary` vrátí **hranici** polygonu (jako linii),
* `shapely.distance` spočítá **nejkratší vzdálenost** (v metrech),
* `nearest_points` najde **nejbližší bod** na hranici – použijeme ho pro nakreslení spojnice.

In [ ]:
hranice = cr.boundary

# vzdálenost všech měst od hranice najednou, převedená na km
mesta["od_hranice_km"] = (shapely.distance(mesta.geometry, hranice) / 1_000).round(1)

fig, ax = plt.subplots(figsize=(11, 6))
plot_polygon(cr, ax=ax, add_points=False, color="whitesmoke", edgecolor="black")

for nazev, mesto in zip(mesta["name"], mesta.geometry):
    bod_na_hranici = nearest_points(mesto, hranice)[1]                   # nejbližší bod na hranici
    spojnice = LineString([mesto, bod_na_hranici])                       # čára mezi městem a hranicí
    plot_line(spojnice, ax=ax, color="tab:red", linewidth=1, add_points=False)
    plot_points(mesto, ax=ax, color="black", markersize=4)
    ax.annotate(nazev, (mesto.x, mesto.y), xytext=(4, 4), textcoords="offset points", fontsize=8)

ax.set_title("Nejkratší spojnice měst se státní hranicí")
ax.set_aspect("equal")
ax.axis("off")
plt.show()

mesta[["name", "od_hranice_km"]].sort_values("od_hranice_km")

## 7. Řeky a města u řek

1. Řeky ořízneme hranicí ČR – **průnik** (`intersection`).
2. Kolem řek vytvoříme pásmo 5 km – **buffer**.
3. Pásma sloučíme do jedné plochy – **sjednocení** (`union_all`).
4. Zjistíme, která města v pásmu leží – `contains`.

In [ ]:
# 1. průnik řek s územím ČR (u řek mimo ČR vznikne prázdná geometrie, ty vyřadíme)
reky_cr = shapely.intersection(reky.geometry, cr)
reky_cr = reky_cr[~shapely.is_empty(reky_cr)]

# 2. + 3. pásmo 5 000 m kolem každé řeky a sloučení všech pásem do jedné plochy
pasmo = shapely.union_all(shapely.buffer(reky_cr, 5_000))

# 4. která města leží v pásmu
u_reky = shapely.contains(pasmo, mesta.geometry)
print("Města u řeky:", ", ".join(mesta["name"][u_reky]))
print("Města mimo:  ", ", ".join(mesta["name"][~u_reky]))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
plot_polygon(cr, ax=ax, add_points=False, color="whitesmoke", edgecolor="black")
plot_polygon(pasmo.intersection(cr), ax=ax, add_points=False, color="lightblue")     # pásmo uvnitř ČR
for reka in reky_cr:
    plot_line(reka, ax=ax, color="tab:blue", linewidth=1, add_points=False)
plot_points(mesta.geometry[u_reky], ax=ax, color="tab:green", markersize=6)           # města u řeky
plot_points(mesta.geometry[~u_reky], ax=ax, color="tab:red", markersize=6)            # města mimo
ax.set_title("Pásmo 5 km kolem řek (zeleně města u řeky, červeně mimo)")
ax.set_aspect("equal")
ax.axis("off")
plt.show()

## 8. Kam umístit popisek?

`centroid` je geometrické **těžiště**. U tvarů jako „podkova“ ale může ležet **mimo** polygon – typickým příkladem je Chorvatsko. Pro popisky v mapě je proto lepší `representative_point()`, který vždy leží uvnitř.

In [ ]:
chorvatsko = staty.loc[staty["ADM0_A3"] == "HRV", "geometry"].iloc[0]

teziste = chorvatsko.centroid
reprezentativni_bod = chorvatsko.representative_point()

print("Leží těžiště v Chorvatsku?             ", chorvatsko.contains(teziste))
print("Leží reprezentativní bod v Chorvatsku? ", chorvatsko.contains(reprezentativni_bod))

fig, ax = plt.subplots(figsize=(6, 5))
plot_polygon(chorvatsko, ax=ax, add_points=False, color="lightgrey", edgecolor="black")
plot_points(teziste, ax=ax, color="red", markersize=10, label="centroid")
plot_points(reprezentativni_bod, ax=ax, color="green", markersize=10, label="representative_point")
ax.legend(loc="lower left")
ax.set_title("Chorvatsko")
ax.set_aspect("equal")
ax.axis("off")
plt.show()

## 9. Celý svět najednou: prostorový index STRtree

Úloha: ke **každému** ze ~7 300 sídel světa najít region, ve kterém leží (kraj, spolkový stát, provincii…, v datech je jich ~4 600). Kdybychom testovali každé sídlo proti každému regionu, bylo by to přes 33 milionů testů.

**STRtree** si regiony uspořádá podle jejich obalových obdélníků. Pro každé sídlo pak rychle najde jen pár regionů, které připadají v úvahu, a přesný test udělá jen pro ně.

In [ ]:
import time                                           # pro měření doby výpočtu

body = mesta_svet.geometry                            # 7 300 bodů
oblasti = regiony.geometry                            # 4 600 polygonů

# a) se STRtree
start = time.perf_counter()
strom = shapely.STRtree(oblasti)                                  # postavení indexu
idx_bod, idx_oblast = strom.query(body, predicate="within")       # dvojice (sídlo, region)
doba_strom = time.perf_counter() - start

# b) bez indexu: každé sídlo proti každému regionu – změříme jen 50 sídel a přepočteme na všechna
start = time.perf_counter()
for bod in body[:50]:
    for oblast in oblasti:
        oblast.contains(bod)
doba_bez_indexu = (time.perf_counter() - start) / 50 * len(body)

print(f"Se STRtree:  {doba_strom:.2f} s")
print(f"Bez indexu:  {doba_bez_indexu:.1f} s (odhad)")
print(f"STRtree je cca {doba_bez_indexu / doba_strom:.0f}× rychlejší")

In [ ]:
# ukázka výsledku: náhodně vybraných 8 sídel a region, ve kterém leží
pd.DataFrame({
    "sídlo": mesta_svet["name"].to_numpy()[idx_bod],
    "region": regiony["name"].to_numpy()[idx_oblast],
    "stát": regiony["admin"].to_numpy()[idx_oblast],
}).sample(8, random_state=1)

## 10. Zjednodušení hranice

`simplify` odstraní vrcholy, které nejsou pro tvar podstatné (algoritmus Douglas–Peucker). Parametr **tolerance** určuje, o kolik metrů se zjednodušená hranice smí odchýlit od původní. Hodí se to například pro webové mapy, kde podrobná hranice zbytečně zpomaluje načítání.

In [ ]:
tolerance = [0, 1_000, 5_000, 20_000]          # v metrech

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, tol in zip(axes, tolerance):
    zjednodusena = cr.simplify(tol)
    pocet_vrcholu = shapely.get_num_coordinates(zjednodusena)
    plot_polygon(zjednodusena, ax=ax, add_points=False, color="tab:purple", alpha=0.5)
    ax.set_title(f"tolerance {tol / 1000:.0f} km\n{pocet_vrcholu} vrcholů")
    ax.set_aspect("equal")
    ax.axis("off")
plt.show()

## Zdroje

[Dokumentace Shapely](https://shapely.readthedocs.io/) · [GEOS](https://libgeos.org/) · [GeoPandas](https://geopandas.org/) · [Natural Earth](https://www.naturalearthdata.com/)